# Chapter 3 — Lists, Loops and Decisions

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch03_lists_loops_decisions.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 3.1 The rules we will use

In [ ]:
def inr(amount):
    """Format a rupee amount with Indian digit grouping: 12,34,567."""
    if amount == float("inf"):
        return "no limit"
    n = int(round(amount))
    text, rest = f"{n % 1000:03d}", n // 1000
    while rest > 0:
        text, rest = f"{rest % 100:02d}," + text, rest // 100
    return text.lstrip("0") or "0"


print(inr(1275000), inr(60000), inr(0), inr(float("inf")))

## 3.2 Lists: an ordered collection

In [ ]:
limits = [400000, 800000, 1200000, 1600000, 2000000, 2400000]
rates = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]
print(limits)
print("Number of limits:", len(limits))

In [ ]:
print("First limit:", limits[0])
print("Third limit:", limits[2])
print("Last limit: ", limits[-1])

In [ ]:
print(limits[0:3])      # items 0, 1, 2
print(limits[:2])       # the first two
print(limits[3:])       # from the fourth to the end
print(limits[::2])      # every second item

In [ ]:
limits.append(float("inf"))
rates.append(0.30)
print(limits)
print(rates)
print("Highest rate:", max(rates), "| number of slabs:", len(rates))

In [ ]:
print(0.15 in rates)
print(0.12 in rates)

## 3.3 Tuples: values that belong together

In [ ]:
slab = (800000, 0.05)
print(slab[0], slab[1])

upper, rate = slab          # "unpacking" a tuple into two names
print(f"Rate {rate:.0%} up to Rs {inr(upper)}")

In [ ]:
INF = float("inf")

NEW_SLABS = [(400000, 0.00), (800000, 0.05), (1200000, 0.10),
             (1600000, 0.15), (2000000, 0.20), (2400000, 0.25), (INF, 0.30)]

OLD_SLABS = [(250000, 0.00), (500000, 0.05), (1000000, 0.20), (INF, 0.30)]

print(len(NEW_SLABS), "new-regime slabs;", len(OLD_SLABS), "old-regime slabs")

## 3.4 Dictionaries: rules with names

In [ ]:
RULES = {
    "new": {
        "slabs": NEW_SLABS,
        "standard_deduction": 75000,
        "rebate_limit": 1200000,       # total income up to which rebate applies
        "max_rebate": 60000,
        "marginal_relief": True,
    },
    "old": {
        "slabs": OLD_SLABS,
        "standard_deduction": 50000,
        "rebate_limit": 500000,
        "max_rebate": 12500,
        "marginal_relief": False,
    },
}
CESS = 0.04                            # Health and Education Cess

print(RULES["new"]["standard_deduction"])
print(RULES["old"]["rebate_limit"])

In [ ]:
print(RULES["new"].get("surcharge", "not modelled"))
print(list(RULES["new"].keys()))

## 3.5 `for` loops: slab by slab

In [ ]:
for upper, rate in NEW_SLABS:
    print(f"up to {inr(upper):>12}: {rate:.0%}")

In [ ]:
uppers = [upper for upper, rate in NEW_SLABS]
lowers = [0] + uppers[:-1]           # 0, then every upper limit except the last

for number, (low, high) in enumerate(zip(lowers, uppers), start=1):
    print(f"Slab {number}: Rs {inr(low):>9} to {inr(high):>12}")

In [ ]:
income = 1000000
tax = 0
lower = 0
for upper, rate in NEW_SLABS:
    if income > lower:                         # does any income reach this slab?
        in_slab = min(income, upper) - lower   # the part of income inside it
        tax = tax + in_slab * rate
        print(f"{rate:4.0%} on Rs {inr(in_slab):>9}  -> tax so far Rs {inr(tax):>9}")
    lower = upper                              # the next slab starts here
print("Tax before rebate and cess:", tax)

In [ ]:
def slab_tax(total_income, slabs):
    """Tax on total_income under a list of (upper limit, rate) slabs."""
    tax = 0
    lower = 0
    for upper, rate in slabs:
        if total_income > lower:
            tax += (min(total_income, upper) - lower) * rate
        lower = upper
    return tax


print(slab_tax(1000000, NEW_SLABS))
print(slab_tax(1000000, OLD_SLABS))

## 3.6 Decisions: `if`, `elif`, `else`

In [ ]:
def describe_slab(total_income):
    """Which new-regime slab does the top rupee of income fall in?"""
    if total_income <= 400000:
        return "nil slab"
    elif total_income <= 1200000:
        return "5% or 10% slab"
    elif total_income <= 2400000:
        return "15% to 25% slab"
    else:
        return "30% slab"


for y in [300000, 900000, 1500000, 3000000]:
    print(f"Rs {inr(y):>9}: {describe_slab(y)}")

### The rebate and marginal relief

In [ ]:
def rebate(total_income, tax, rules):
    """Rebate under section 156 (Income-tax Act, 2025), resident individual."""
    limit = rules["rebate_limit"]
    if total_income <= limit:
        return min(tax, rules["max_rebate"])
    elif rules["marginal_relief"] and tax > total_income - limit:
        return tax - (total_income - limit)
    else:
        return 0


new = RULES["new"]
for y in [1200000, 1210000, 1250000, 1300000]:
    t = slab_tax(y, NEW_SLABS)
    print(f"Income Rs {inr(y):>9}: slab tax {inr(t):>7}, rebate {inr(rebate(y, t, new)):>7}, "
          f"tax after rebate {inr(t - rebate(y, t, new)):>7}")

## 3.7 The complete calculator

In [ ]:
def total_income(gross_salary, regime, deductions=0):
    """Taxable income: salary minus standard deduction (and, in the old
    regime, other deductions such as those in old section 80C).
    Never below zero."""
    rules = RULES[regime]
    if regime == "new":
        deductions = 0          # the new regime allows almost no deductions
    return max(gross_salary - rules["standard_deduction"] - deductions, 0)


def income_tax(gross_salary, regime, deductions=0):
    """Income tax plus cess (Rs) for a resident individual below 60
    whose only income is a salary. Tax year 2026-27 rules; no surcharge,
    so use only for incomes up to Rs 50 lakh."""
    rules = RULES[regime]
    ti = total_income(gross_salary, regime, deductions)
    tax = slab_tax(ti, rules["slabs"])
    tax = tax - rebate(ti, tax, rules)
    return round(tax * (1 + CESS))


print(income_tax(1500000, "new"))
print(income_tax(1500000, "old", deductions=200000))

In [ ]:
salary = 1500000
taxes = {}                       # an empty dictionary, filled in by the loop
for regime in ["old", "new"]:
    d = 200000 if regime == "old" else 0
    taxes[regime] = income_tax(salary, regime, deductions=d)

print(taxes)
best = min(taxes, key=taxes.get)
print(f"Priya should choose the {best} regime and save "
      f"Rs {inr(abs(taxes['old'] - taxes['new']))}")

## 3.8 Check against the official examples

In [ ]:
budget_table = {        # income: (tax on slabs, tax after rebate), Rs
    800000: (20000, 0),
    900000: (30000, 0),
    1000000: (40000, 0),
    1100000: (50000, 0),
    1200000: (60000, 0),
    1600000: (120000, 120000),
    2000000: (200000, 200000),
    2400000: (300000, 300000),
    5000000: (1080000, 1080000),
}

print(f"{'income':>10} {'slab tax':>10} {'ours':>10} {'after rebate':>13} {'ours':>10}")
all_match = True
for ti, (pub_tax, pub_after) in budget_table.items():
    our_tax = slab_tax(ti, NEW_SLABS)
    our_after = our_tax - rebate(ti, our_tax, RULES["new"])
    all_match = all_match and our_tax == pub_tax and our_after == pub_after
    print(f"{inr(ti):>10} {inr(pub_tax):>10} {inr(our_tax):>10} {inr(pub_after):>13} {inr(our_after):>10}")
print("Every value matches:", all_match)

In [ ]:
SLABS_2024 = [(300000, 0.00), (700000, 0.05), (1000000, 0.10),
              (1200000, 0.15), (1500000, 0.20), (INF, 0.30)]
print("Slab tax on Rs 12 lakh under 2024-25 slabs:",
      f"{inr(slab_tax(1200000, SLABS_2024))}")

## 3.9 Comprehensions: a whole schedule in one line

In [ ]:
salaries = []
for s in range(0, 5000001, 25000):        # Rs 0 to 50 lakh in steps of 25,000
    salaries.append(s)
print(len(salaries), "salaries, from", salaries[0], "to", salaries[-1])

In [ ]:
salaries = [s for s in range(0, 5000001, 25000)]
tax_new = [income_tax(s, "new") for s in salaries]
tax_old = [income_tax(s, "old") for s in salaries]
tax_old_3l = [income_tax(s, "old", deductions=300000) for s in salaries]

print("Salary Rs 20 lakh is at index", salaries.index(2000000))
i = salaries.index(2000000)
print(f"New: {inr(tax_new[i])}  Old, no deductions: {inr(tax_old[i])}  "
      f"Old, Rs 3 lakh deductions: {inr(tax_old_3l[i])}")

In [ ]:
zero_tax = [s for s, t in zip(salaries, tax_new) if t == 0]
print(f"Highest salary in our grid with zero tax (new regime): Rs {inr(max(zero_tax))}")

In [ ]:
import matplotlib.pyplot as plt

lakhs = [s / 1e5 for s in salaries]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(lakhs, [t / 1e5 for t in tax_old], color="#c55a11", linewidth=2,
        label="Old regime, no deductions")
ax.plot(lakhs, [t / 1e5 for t in tax_old_3l], color="#c55a11", linewidth=2,
        linestyle="--", label="Old regime, Rs 3 lakh of deductions")
ax.plot(lakhs, [t / 1e5 for t in tax_new], color="#1f4e79", linewidth=2.4,
        label="New regime")
ax.set_xlabel("Gross salary, Rs lakh")
ax.set_ylabel("Income tax + cess, Rs lakh")
ax.set_title("Income tax by salary, tax year 2026-27", loc="left", fontsize=12)
ax.legend(frameon=False, loc="upper left")
ax.set_xlim(0, 50)
ax.set_ylim(0, None)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Resident individual below 60, salary income only, no "
         "surcharge.\nSource: author's calculation; Income-tax Act, 2025 "
         "(ss. 19, 156, 202) and Finance Bill, 2026.", fontsize=8, color="grey")
plt.show()

### A closer look at marginal relief

In [ ]:
no_relief = dict(RULES["new"])              # a copy, so RULES is untouched
no_relief["marginal_relief"] = False


def tax_without_relief(gross):
    ti = total_income(gross, "new")
    t = slab_tax(ti, NEW_SLABS)
    return round((t - rebate(ti, t, no_relief)) * (1 + CESS))


zoom = list(range(1200000, 1400001, 1000))
actual = [income_tax(s, "new") for s in zoom]
without = [tax_without_relief(s) for s in zoom]

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot([s / 1e5 for s in zoom], without, color="#9db7d5", linewidth=2,
        label="Without marginal relief")
ax.plot([s / 1e5 for s in zoom], actual, color="#1f4e79", linewidth=2.4,
        label="Actual (with marginal relief)")
ax.axvline(12.75, color="grey", linestyle=":", linewidth=1)
ax.text(12.77, 80000, "Rs 12.75 lakh", color="grey", fontsize=9)
ax.set_xlabel("Gross salary, Rs lakh")
ax.set_ylabel("Income tax + cess, Rs")
ax.yaxis.set_major_formatter(lambda value, position: inr(value))
ax.set_title("New regime: how marginal relief removes the tax cliff",
             loc="left", fontsize=12)
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: author's calculation from section 156(2) "
         "of the Income-tax Act, 2025.", fontsize=8, color="grey")
plt.show()

In [ ]:
ramp_end = [s for s, a, w in zip(zoom, actual, without) if a == w and s > 1275000][0]
print(f"Marginal relief ends at a salary of about Rs {inr(ramp_end)}")
step_up = without[zoom.index(1276000)] - without[zoom.index(1275000)]
print(f"Without relief, Rs 1,000 more salary at the threshold costs Rs {inr(step_up)} in tax")

## 3.10 `while` loops: finding the break-even deduction

In [ ]:
def break_even(gross_salary, step=1000):
    """Smallest deduction (in steps of `step`) at which the old regime's
    tax is no higher than the new regime's."""
    target = income_tax(gross_salary, "new")
    d = 0
    while income_tax(gross_salary, "old", deductions=d) > target:
        d += step
    return d


print(f"Priya (Rs 15 lakh): break-even deductions Rs {inr(break_even(1500000))}")

In [ ]:
grid = [s for s in salaries if s >= 500000]
needed = [break_even(s) for s in grid]

for s in [700000, 1000000, 1275000, 1500000, 2000000, 2400000, 3000000, 5000000]:
    d = needed[grid.index(s)]
    print(f"Salary Rs {s / 1e5:5.2f} lakh: old regime needs deductions of "
          f"Rs {d / 1e5:4.2f} lakh ({d / s:5.1%} of salary)")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.fill_between([s / 1e5 for s in grid], [d / 1e5 for d in needed], 12,
                color="#c55a11", alpha=0.12)
ax.plot([s / 1e5 for s in grid], [d / 1e5 for d in needed], color="#1f4e79",
        linewidth=2.4)
ax.text(36, 9.5, "Old regime better", color="#c55a11", fontsize=11)
ax.axvline(12.75, color="grey", linestyle=":", linewidth=1)
ax.text(13.2, 10.6, "← left of this line (salary up to Rs 12.75 lakh)\n"
        "    the regimes can at best tie at zero tax", color="grey", fontsize=9)
ax.text(36, 1.5, "New regime better", color="#1f4e79", fontsize=11)
ax.set_ylim(0, 12)
ax.set_xlim(5, 50)
ax.set_xlabel("Gross salary, Rs lakh")
ax.set_ylabel("Deductions needed, Rs lakh")
ax.set_title("Break-even deductions: when does the old regime win?",
             loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.06, "Deductions claimable only in the old regime (e.g. "
         "old sections 80C, 80D, HRA), beyond the standard deduction.\n"
         "Tax year 2026-27 rules. Source: author's calculation.", fontsize=8,
         color="grey")
plt.show()

In [ ]:
peak = max(needed[:grid.index(1500000)])
print(f"Peak: Rs {inr(peak)} at a salary of Rs {inr(grid[needed.index(peak)])}")

window = needed[grid.index(1275000):grid.index(1600000)]
dip = min(window)
print(f"Dip:  Rs {inr(dip)} at a salary of Rs {inr(grid[needed.index(dip)])}")

plateau = max(needed)
print(f"Plateau: Rs {inr(plateau)} from a salary of Rs "
      f"{inr(grid[needed.index(plateau)])} upwards")

for s in [3000000, 5000000]:
    gap = income_tax(s, "old") - income_tax(s, "new")
    print(f"Extra tax in the old regime with no deductions at Rs {inr(s)}: "
          f"Rs {inr(gap)}")

## 3.11 Putting it together: a regime adviser

In [ ]:
def advise(gross_salary, deductions):
    """Compare both regimes for a salary and old-regime deductions."""
    if gross_salary > 5000000:
        return "Above Rs 50 lakh: surcharge applies, which this calculator ignores."
    tax = {"old": income_tax(gross_salary, "old", deductions),
           "new": income_tax(gross_salary, "new")}
    lines = [f"Salary Rs {inr(gross_salary)}, old-regime deductions Rs {inr(deductions)}"]
    for regime, amount in tax.items():
        lines.append(f"  {regime} regime: tax Rs {inr(amount):>9}"
                     f"  ({amount / gross_salary:.1%} of salary)")
    if tax["old"] < tax["new"]:
        lines.append(f"  -> old regime saves Rs {inr(tax['new'] - tax['old'])}")
    elif tax["new"] < tax["old"]:
        lines.append(f"  -> new regime saves Rs {inr(tax['old'] - tax['new'])}")
    else:
        lines.append("  -> both regimes give the same tax")
    return "\n".join(lines)


print(advise(1000000, 150000))
print(advise(3000000, 500000))
print(advise(6000000, 0))

## Key ideas

## Exercises

## Sources of the tax rules

## Further reading